# Triangular Arbitrage System - Google Colab Implementation

## Overview

This document provides comprehensive documentation for the Triangular Arbitrage System implementation. The system simulates and analyzes triangular arbitrage opportunities in the forex market, focusing on EUR/USD, USD/JPY, and EUR/JPY currency pairs.

## System Components

The implementation consists of the following components:

1. **Exchange Rate Modeling** (`triangular_arbitrage_implementation.py`): Handles exchange rate data generation, log return calculations, and basic visualization.

2. **Parameter Estimation** (`parameter_estimation.py`): Estimates volatility parameters, correlation coefficients, and mispricing volatility from exchange rate data.

3. **Mispricing Calculation** (`mispricing_calculation.py`): Calculates implied cross rates, generates mispricing terms, and determines actual rates.

4. **Arbitrage Detection** (`arbitrage_detection.py`): Identifies arbitrage opportunities, calculates theoretical profits, and analyzes feasibility considering transaction costs.

5. **Visualization** (`visualization.py`): Provides comprehensive visualization tools for analyzing the triangular arbitrage system.

6. **Main Script** (`main.py`): Integrates all components and provides functions for running simulations, testing with different parameters, and validating against examples.

## Mathematical Model

The implementation follows the mathematical model described in the project documentation:

### Exchange Rate Notation
- **EUR/USD (R_EUR/USD)**: Price of 1 EUR in terms of USD
- **USD/JPY (R_USD/JPY)**: Price of 1 USD in terms of JPY
- **EUR/JPY (R_EUR/JPY)**: Price of 1 EUR in terms of JPY

### Implied Cross Rate
The implied cross rate for EUR/JPY is calculated as:
```
R_EUR/JPY^implied = R_EUR/USD × R_USD/JPY
```

### Arbitrage Opportunity
An arbitrage opportunity exists when the actual quoted rate deviates from the implied rate:
```
R_EUR/JPY^actual ≠ R_EUR/JPY^implied
```

### Profit Factor
The profit factor Π (representing the factor by which the initial capital is multiplied) is:
```
Π = max{ R_EUR/JPY^actual / (R_EUR/USD × R_USD/JPY), (R_EUR/USD × R_USD/JPY) / R_EUR/JPY^actual }
```

### Stochastic Exchange Rate Modeling
Short-term changes (logarithmic returns) of the two "base" pairs (EUR/USD and USD/JPY) are modeled as random variables from Normal distributions:
```
ΔrEU ~ N(0, σ²EU)   (Log-return for EUR/USD)
ΔrUJ ~ N(0, σ²UJ)   (Log-return for USD/JPY)
Corr(ΔrEU, ΔrUJ) = ρ   (Correlation between the returns)
```

### Mispricing Term
The mispricing term ε is modeled as a small, random fluctuation, normally distributed around zero:
```
R_EUR/JPY^actual = R_EUR/JPY^implied · (1 + ε)
ε ~ N(0, σ²ε)
```

### Transaction Costs
A triangular arbitrage opportunity is only truly exploitable in practice if:
```
Profit(ε) > Total Transaction Costs
```

## Usage Guide

### Running a Basic Simulation

To run a basic triangular arbitrage simulation:

```python
from main import run_triangular_arbitrage_simulation

# Run simulation with default parameters
exchange_data, estimator, simulator, detector, visualizer = run_triangular_arbitrage_simulation(
    num_periods=1000,
    transaction_cost=0.0005,  # 0.05% per trade
    output_pdf="triangular_arbitrage_report.pdf"
)

# Display visualizations
fig1 = visualizer.create_exchange_rate_dashboard()
plt.figure(fig1.number)
plt.show()
```

### Testing with Different Parameters

To test the system with different parameter settings:

```python
from main import test_with_different_parameters

# Run tests with various transaction costs, volatilities, and correlations
test_with_different_parameters()
```

### Validating Against Excel Example

To validate the implementation against the Excel example:

```python
from main import validate_with_excel_example

# Run validation
exchange_data, estimator, simulator, detector, visualizer = validate_with_excel_example()
```

## Parameter Descriptions

The system accepts the following parameters:

- **num_periods**: Number of periods for simulation (default: 1000)
- **transaction_cost**: Transaction cost as a percentage (default: 0.0005 for 0.05%)
- **volatility_epsilon**: Volatility of the mispricing term (default: 0.001)
- **correlation**: Correlation between EUR/USD and USD/JPY returns (default: -0.84)
- **initial_rates**: Initial exchange rates (default: {'EUR/USD': 1.13, 'USD/JPY': 143.0, 'EUR/JPY': 161.5})
- **volatilities**: Volatility parameters for exchange rates (default: {'EUR/USD': 0.004, 'USD/JPY': 0.0043})
- **output_pdf**: Path to save the report in PDF format (optional)

## Implementation Details

### ExchangeRateData Class

The `ExchangeRateData` class handles exchange rate data generation and management:

- Generates synthetic exchange rate data with specified parameters
- Calculates logarithmic returns
- Provides methods for accessing and visualizing exchange rates

```python
# Create exchange rate data
exchange_data = ExchangeRateData(
    data_source='synthetic',
    num_periods=1000,
    initial_rates={'EUR/USD': 1.13, 'USD/JPY': 143.0, 'EUR/JPY': 161.5},
    volatilities={'EUR/USD': 0.004, 'USD/JPY': 0.0043},
    correlation=-0.84
)

# Get rates and returns
rates_df = exchange_data.get_rates()
returns_df = exchange_data.get_returns()

# Plot rates
exchange_data.plot_rates()
```

### ParameterEstimator Class

The `ParameterEstimator` class estimates statistical parameters from exchange rate data:

- Estimates volatility parameters for EUR/USD and USD/JPY
- Estimates correlation between EUR/USD and USD/JPY returns
- Estimates volatility of the mispricing term

```python
# Create parameter estimator
estimator = ParameterEstimator(exchange_data)

# Estimate all parameters
parameters = estimator.estimate_all_parameters()

# Plot parameter stability
estimator.plot_parameter_stability()
```

### MispricingSimulator Class

The `MispricingSimulator` class simulates mispricing in cross rates:

- Calculates implied cross rates
- Generates mispricing terms based on normal distribution
- Calculates actual rates incorporating mispricing

```python
# Create mispricing simulator
simulator = MispricingSimulator(
    exchange_data,
    parameters={'volatility_epsilon': 0.001}
)

# Calculate implied rates
implied_rates = simulator.calculate_implied_rates()

# Generate mispricing
epsilon = simulator.generate_mispricing()

# Calculate actual rates
actual_rates = simulator.calculate_actual_rates()

# Analyze mispricing distribution
stats = simulator.analyze_mispricing_distribution()
```

### ArbitrageDetector Class

The `ArbitrageDetector` class detects arbitrage opportunities and calculates profits:

- Detects arbitrage opportunities based on rate discrepancies
- Calculates theoretical profits from identified opportunities
- Analyzes feasibility considering transaction costs

```python
# Create arbitrage detector
detector = ArbitrageDetector(
    exchange_data,
    mispricing_simulator=simulator,
    transaction_costs=0.0005  # 0.05% per trade
)

# Detect opportunities
detector.detect_opportunities()

# Calculate profits
detector.calculate_profits()

# Analyze feasibility
detector.analyze_feasibility()

# Get results
results_df, feasibility_summary = detector.get_results()
```

### ArbitrageVisualizer Class

The `ArbitrageVisualizer` class provides comprehensive visualization tools:

- Creates integrated dashboards of arbitrage results
- Generates publication-quality figures for reports
- Visualizes relationships between different parameters

```python
# Create visualizer
visualizer = ArbitrageVisualizer(
    exchange_data,
    parameter_estimator=estimator,
    mispricing_simulator=simulator,
    arbitrage_detector=detector
)

# Create exchange rate dashboard
visualizer.create_exchange_rate_dashboard()

# Create arbitrage dashboard
visualizer.create_arbitrage_dashboard()

# Create parameter dashboard
visualizer.create_parameter_dashboard()

# Create transaction cost analysis
visualizer.create_transaction_cost_analysis()

# Create comprehensive report
visualizer.create_comprehensive_report(output_file="triangular_arbitrage_report.pdf")
```


## Conclusion

This triangular arbitrage system provides a comprehensive implementation of the mathematical model described in the project documentation. It allows for simulation, analysis, and visualization of triangular arbitrage opportunities in the forex market, with consideration for transaction costs and market microstructure effects.

The system is designed to be flexible and extensible, allowing for testing with different parameters and validation against real-world examples. The comprehensive visualization tools provide insights into the behavior of the system and the characteristics of arbitrage opportunities.

In [ ]:
"""
Implementation of Triangular Arbitrage System

This notebook implements a complete triangular arbitrage system for forex markets,
focusing on EUR/USD, USD/JPY, and EUR/JPY currency pairs.

The implementation follows the mathematical modeling described in the project documentation
and includes components for exchange rate modeling, parameter estimation, mispricing calculation,
arbitrage detection, and visualization.
"""

# Install required packages
!pip install numpy pandas matplotlib seaborn

# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import os
from typing import Dict, List, Tuple, Optional, Union
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
from matplotlib.backends.backend_pdf import PdfPages
import requests

# Set random seed for reproducibility
np.random.seed(42)

# Configure matplotlib
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

In [ ]:
#############################################
# Part 1: Exchange Rate Data Implementation #
#############################################

class ExchangeRateData:
    """
    Handles exchange rate data (historical or synthetic)

    This class provides functionality to:
    1. Generate synthetic exchange rate data
    2. Calculate logarithmic returns
    3. Manage and access exchange rate time series
    """

    def __init__(self, data_source='synthetic', num_periods=1000,
                 initial_rates=None, volatilities=None, correlation=None):
        """
        Initialize the exchange rate data handler

        Parameters:
        -----------
        data_source : str
            Source of data ('synthetic' or 'historical')
        num_periods : int
            Number of periods for synthetic data generation
        initial_rates : dict
            Initial exchange rates for synthetic data generation
            Format: {'EUR/USD': float, 'USD/JPY': float, 'EUR/JPY': float}
        volatilities : dict
            Volatility parameters for synthetic data generation
            Format: {'EUR/USD': float, 'USD/JPY': float}
        correlation : float
            Correlation between EUR/USD and USD/JPY returns
        """
        self.data_source = data_source
        self.num_periods = num_periods

        # Default initial rates if not provided
        self.initial_rates = initial_rates or {
            'EUR/USD': 1.13,
            'USD/JPY': 143.0,
            'EUR/JPY': 161.5
        }

        # Default volatilities if not provided
        self.volatilities = volatilities or {
            'EUR/USD': 0.004,  # σEU
            'USD/JPY': 0.0043  # σUJ
        }

        # Default correlation if not provided
        self.correlation = -0.84 if correlation is None else correlation

        # Initialize data structures
        self.rates_df = None
        self.returns_df = None

        # Generate or load data based on source
        if data_source == 'synthetic':
            self.generate_synthetic_data()
        else:
            self.fetch_historical_data()

    def fetch_historical_data(self) -> pd.DataFrame:
        url = "https://palashsharma.com/exchange_rate_cache.json"
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()

        df = (pd.DataFrame.from_dict(resp.json(), orient="index")
                .rename_axis("Date")
                .reset_index()
                .assign(Date=lambda d: pd.to_datetime(d["Date"], format="%Y/%m/%d"))
                .sort_values("Date")
                .set_index("Date"))

        df["EUR/JPY_implied"] = df["EUR/USD"] * df["USD/JPY"]
        df["Epsilon"] = df["EUR/JPY"] / df["EUR/JPY_implied"] - 1

        self.rates_df = df.tail(self.num_periods)
        self.calculate_log_returns()
        return self.rates_df

    def generate_synthetic_data(self):
        """
        Generate synthetic exchange rate data with specified parameters

        The generation follows these steps:
        1. Create a time index
        2. Generate correlated normal random variables for log returns
        3. Convert log returns to exchange rates
        4. Calculate the implied EUR/JPY rate
        5. Add mispricing to create actual EUR/JPY rate
        """
        # Create time index
        dates = [datetime.now() - timedelta(days=i) for i in range(self.num_periods)]
        dates.reverse()

        # Extract parameters
        sigma_eu = self.volatilities['EUR/USD']
        sigma_uj = self.volatilities['USD/JPY']
        rho = self.correlation

        # Generate correlated normal random variables for log returns
        # Using Cholesky decomposition to generate correlated random variables
        cov_matrix = np.array([
            [sigma_eu**2, rho * sigma_eu * sigma_uj],
            [rho * sigma_eu * sigma_uj, sigma_uj**2]
        ])

        L = np.linalg.cholesky(cov_matrix)
        uncorrelated_returns = np.random.normal(0, 1, size=(2, self.num_periods))
        correlated_returns = np.dot(L, uncorrelated_returns)

        # Extract the correlated returns
        log_returns_eu = correlated_returns[0]
        log_returns_uj = correlated_returns[1]

        # Initialize arrays for rates
        rates_eu = np.zeros(self.num_periods)
        rates_uj = np.zeros(self.num_periods)

        # Set initial rates
        rates_eu[0] = self.initial_rates['EUR/USD']
        rates_uj[0] = self.initial_rates['USD/JPY']

        # Generate rates from log returns
        for i in range(1, self.num_periods):
            rates_eu[i] = rates_eu[i-1] * np.exp(log_returns_eu[i-1])
            rates_uj[i] = rates_uj[i-1] * np.exp(log_returns_uj[i-1])

        # Calculate implied EUR/JPY rate
        implied_rates_ej = rates_eu * rates_uj

        # Add mispricing to create actual EUR/JPY rate
        # Mispricing is modeled as a normal random variable with mean 0 and std dev sigma_epsilon
        sigma_epsilon = 0.001  # Default mispricing volatility
        epsilon = np.random.normal(0, sigma_epsilon, self.num_periods)
        actual_rates_ej = implied_rates_ej * (1 + epsilon)

        # Create DataFrame
        self.rates_df = pd.DataFrame({
            'Date': dates,
            'EUR/USD': rates_eu,
            'USD/JPY': rates_uj,
            'EUR/JPY_implied': implied_rates_ej,
            'EUR/JPY': actual_rates_ej,
            'Epsilon': epsilon
        })

        self.rates_df.set_index('Date', inplace=True)

        # Calculate log returns for analysis
        self.calculate_log_returns()

        return self.rates_df

    def calculate_log_returns(self):
        """
        Calculate logarithmic returns for all currency pairs

        For each currency pair, the log return at time t is:
        r(t) = ln(Rate(t) / Rate(t-1))
        """
        if self.rates_df is None:
            raise ValueError("No exchange rate data available. Generate or load data first.")

        # Create a new DataFrame for returns
        self.returns_df = pd.DataFrame(index=self.rates_df.index)

        # Calculate log returns for each currency pair
        for pair in ['EUR/USD', 'USD/JPY', 'EUR/JPY', 'EUR/JPY_implied']:
            if pair in self.rates_df.columns:
                self.returns_df[f'Log_r_{pair}'] = np.log(
                    self.rates_df[pair] / self.rates_df[pair].shift(1)
                )

        # Drop the first row with NaN values
        self.returns_df = self.returns_df.dropna()

        return self.returns_df

    def get_rates(self):
        """Return the exchange rates DataFrame"""
        return self.rates_df

    def get_returns(self):
        """Return the log returns DataFrame"""
        return self.returns_df

    def plot_rates(self):
        """Plot the exchange rates over time"""
        if self.rates_df is None:
            raise ValueError("No exchange rate data available.")

        fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

        # Plot EUR/USD
        self.rates_df['EUR/USD'].plot(ax=axes[0], color='blue')
        axes[0].set_title('EUR/USD Exchange Rate')
        axes[0].set_ylabel('Rate')
        axes[0].grid(True)

        # Plot USD/JPY
        self.rates_df['USD/JPY'].plot(ax=axes[1], color='green')
        axes[1].set_title('USD/JPY Exchange Rate')
        axes[1].set_ylabel('Rate')
        axes[1].grid(True)

        # Plot EUR/JPY (actual and implied)
        self.rates_df['EUR/JPY'].plot(ax=axes[2], color='red', label='Actual')
        self.rates_df['EUR/JPY_implied'].plot(ax=axes[2], color='orange', linestyle='--', label='Implied')
        axes[2].set_title('EUR/JPY Exchange Rate')
        axes[2].set_ylabel('Rate')
        axes[2].set_xlabel('Date')
        axes[2].legend()
        axes[2].grid(True)

        plt.tight_layout()
        return fig

In [ ]:
#############################################
# Part 2: Parameter Estimation             #
#############################################

class ParameterEstimator:
    """
    Estimates statistical parameters from exchange rate data

    This class provides functionality to:
    1. Estimate volatility parameters for EUR/USD and USD/JPY
    2. Estimate correlation between EUR/USD and USD/JPY returns
    3. Estimate volatility of the mispricing term
    """

    def __init__(self, exchange_rate_data):
        """
        Initialize the parameter estimator

        Parameters:
        -----------
        exchange_rate_data : ExchangeRateData
            Exchange rate data object containing rates and returns
        """
        self.exchange_data = exchange_rate_data
        self.rates_df = exchange_rate_data.get_rates()
        self.returns_df = exchange_rate_data.get_returns()

        # Initialize parameter storage
        self.parameters = {
            'volatility_EUR/USD': None,
            'volatility_USD/JPY': None,
            'correlation': None,
            'volatility_epsilon': None
        }

        # Validate data
        self._validate_data()

    def _validate_data(self):
        """Validate that the required data is available for parameter estimation"""
        if self.returns_df is None:
            raise ValueError("No returns data available. Calculate returns first.")

        required_columns = ['Log_r_EUR/USD', 'Log_r_USD/JPY']
        for col in required_columns:
            if col not in self.returns_df.columns:
                raise ValueError(f"Required column '{col}' not found in returns data.")

        if self.rates_df is None:
            raise ValueError("No exchange rate data available.")

        if 'Epsilon' not in self.rates_df.columns:
            raise ValueError("Mispricing (Epsilon) data not found in exchange rate data.")

    def estimate_volatilities(self, window_size=None):
        """
        Estimate volatility parameters for EUR/USD and USD/JPY

        Parameters:
        -----------
        window_size : int, optional
            Size of the rolling window for volatility estimation.
            If None, uses the entire dataset.
        """
        # If window_size is None, use the entire dataset
        if window_size is None:
            # Calculate volatility for EUR/USD
            self.parameters['volatility_EUR/USD'] = np.std(
                self.returns_df['Log_r_EUR/USD'], ddof=1
            )

            # Calculate volatility for USD/JPY
            self.parameters['volatility_USD/JPY'] = np.std(
                self.returns_df['Log_r_USD/JPY'], ddof=1
            )
        else:
            # Calculate rolling volatility
            rolling_vol_eu = self.returns_df['Log_r_EUR/USD'].rolling(
                window=window_size
            ).std(ddof=1)

            rolling_vol_uj = self.returns_df['Log_r_USD/JPY'].rolling(
                window=window_size
            ).std(ddof=1)

            # Use the latest values
            self.parameters['volatility_EUR/USD'] = rolling_vol_eu.iloc[-1]
            self.parameters['volatility_USD/JPY'] = rolling_vol_uj.iloc[-1]

        return {
            'volatility_EUR/USD': self.parameters['volatility_EUR/USD'],
            'volatility_USD/JPY': self.parameters['volatility_USD/JPY']
        }

    def estimate_correlation(self, window_size=None):
        """
        Estimate correlation between EUR/USD and USD/JPY returns

        Parameters:
        -----------
        window_size : int, optional
            Size of the rolling window for correlation estimation.
            If None, uses the entire dataset.
        """
        # If window_size is None, use the entire dataset
        if window_size is None:
            # Calculate correlation
            self.parameters['correlation'] = np.corrcoef(
                self.returns_df['Log_r_EUR/USD'],
                self.returns_df['Log_r_USD/JPY']
            )[0, 1]
        else:
            # Calculate rolling correlation
            rolling_corr = self.returns_df['Log_r_EUR/USD'].rolling(
                window=window_size
            ).corr(self.returns_df['Log_r_USD/JPY'])

            # Use the latest value
            self.parameters['correlation'] = rolling_corr.iloc[-1]

        return self.parameters['correlation']

    def estimate_mispricing_volatility(self, window_size=None):
        """
        Estimate volatility of the mispricing term

        Parameters:
        -----------
        window_size : int, optional
            Size of the rolling window for volatility estimation.
            If None, uses the entire dataset.
        """
        # If window_size is None, use the entire dataset
        if window_size is None:
            # Calculate mispricing volatility
            self.parameters['volatility_epsilon'] = np.std(
                self.rates_df['Epsilon'], ddof=1
            )
        else:
            # Calculate rolling volatility
            rolling_vol_eps = self.rates_df['Epsilon'].rolling(
                window=window_size
            ).std(ddof=1)

            # Use the latest value
            self.parameters['volatility_epsilon'] = rolling_vol_eps.iloc[-1]

        return self.parameters['volatility_epsilon']

    def estimate_all_parameters(self, window_size=None):
        """
        Estimate all parameters at once

        Parameters:
        -----------
        window_size : int, optional
            Size of the rolling window for parameter estimation.
            If None, uses the entire dataset.
        """
        self.estimate_volatilities(window_size)
        self.estimate_correlation(window_size)
        self.estimate_mispricing_volatility(window_size)

        return self.get_parameters()

    def get_parameters(self):
        """Return all estimated parameters"""
        return self.parameters

In [ ]:
#############################################
# Part 3: Mispricing Calculation           #
#############################################

class MispricingSimulator:
    """
    Simulates mispricing in cross rates

    This class provides functionality to:
    1. Calculate implied cross rates
    2. Generate mispricing terms based on normal distribution
    3. Calculate actual rates incorporating mispricing
    """

    def __init__(self, exchange_rate_data, parameters=None):
        """
        Initialize the mispricing simulator

        Parameters:
        -----------
        exchange_rate_data : ExchangeRateData
            Exchange rate data object containing rates and returns
        parameters : dict, optional
            Statistical parameters for mispricing simulation
            Format: {
                'volatility_epsilon': float,
                'mean_epsilon': float (default: 0)
            }
        """
        self.exchange_data = exchange_rate_data
        self.rates_df = exchange_rate_data.get_rates().copy()

        # Default parameters if not provided
        if parameters is None:
            parameters = {'volatility_epsilon': 0.001}

        # Ensure mean_epsilon exists in parameters with default value 0.0
        if 'mean_epsilon' not in parameters:
            parameters['mean_epsilon'] = 0.0

        self.parameters = parameters

    def calculate_implied_rates(self, rates_df=None):
        """
        Calculate implied cross rates

        Parameters:
        -----------
        rates_df : pandas.DataFrame, optional
            Exchange rate data to use for calculation
            If None, uses the data from exchange_rate_data
        """
        if rates_df is None:
            rates_df = self.rates_df

        # Calculate implied EUR/JPY rate
        implied_rates = rates_df['EUR/USD'] * rates_df['USD/JPY']

        # Store in the rates DataFrame
        rates_df['EUR/JPY_implied'] = implied_rates

        return implied_rates

    def generate_mispricing(self, num_periods=None, custom_volatility=None):
        """
        Generate mispricing terms based on normal distribution

        Parameters:
        -----------
        num_periods : int, optional
            Number of periods to generate mispricing for
            If None, uses the length of the rates DataFrame
        custom_volatility : float, optional
            Custom volatility parameter for mispricing generation
            If None, uses the volatility_epsilon from parameters
        """
        if num_periods is None:
            num_periods = len(self.rates_df)

        # Use custom volatility if provided, otherwise use from parameters
        volatility = custom_volatility or self.parameters['volatility_epsilon']
        mean = self.parameters.get('mean_epsilon', 0.0)  # Default to 0.0 if not present

        # Generate mispricing terms
        epsilon = np.random.normal(mean, volatility, num_periods)

        # Store in the rates DataFrame
        self.rates_df['Epsilon'] = epsilon

        return pd.Series(epsilon, index=self.rates_df.index)

    def calculate_actual_rates(self, implied_rates=None, epsilon=None):
        """
        Calculate actual rates incorporating mispricing

        Parameters:
        -----------
        implied_rates : pandas.Series, optional
            Implied EUR/JPY rates
            If None, calculates from the rates DataFrame
        epsilon : pandas.Series, optional
            Mispricing terms
            If None, uses the Epsilon column from the rates DataFrame
        """
        # Calculate implied rates if not provided
        if implied_rates is None:
            implied_rates = self.calculate_implied_rates()

        # Use existing epsilon if not provided
        if epsilon is None:
            if 'Epsilon' not in self.rates_df.columns:
                epsilon = self.generate_mispricing()
            else:
                epsilon = self.rates_df['Epsilon']

        # Calculate actual rates
        actual_rates = implied_rates * (1 + epsilon)

        # Store in the rates DataFrame
        self.rates_df['EUR/JPY'] = actual_rates

        return actual_rates

    def analyze_mispricing_distribution(self):
        """Analyze the distribution of mispricing terms"""
        if 'Epsilon' not in self.rates_df.columns:
            raise ValueError("No mispricing data available. Generate mispricing first.")

        # Calculate statistics
        epsilon = self.rates_df['Epsilon'].dropna()
        stats = {
            'mean': epsilon.mean(),
            'median': epsilon.median(),
            'std_dev': epsilon.std(),
            'min': epsilon.min(),
            'max': epsilon.max(),
            'skewness': epsilon.skew(),
            'kurtosis': epsilon.kurt(),
            'q1': epsilon.quantile(0.25),
            'q3': epsilon.quantile(0.75)
        }

        return stats

In [ ]:
#############################################
# Part 4: Arbitrage Detection              #
#############################################

class ArbitrageDetector:
    """
    Detects arbitrage opportunities and calculates profits

    This class provides functionality to:
    1. Detect arbitrage opportunities based on rate discrepancies
    2. Calculate theoretical profits from identified opportunities
    3. Analyze feasibility considering transaction costs
    """

    def __init__(self, exchange_rate_data, mispricing_simulator=None, transaction_costs=0.0):
        """
        Initialize the arbitrage detector

        Parameters:
        -----------
        exchange_rate_data : ExchangeRateData
            Exchange rate data object containing rates and returns
        mispricing_simulator : MispricingSimulator, optional
            Mispricing simulator object for accessing mispricing data
        transaction_costs : float or dict, optional
            Transaction costs as a percentage (e.g., 0.001 for 0.1%)
            Can be a single value for all pairs or a dictionary with costs per pair
        """
        self.exchange_data = exchange_rate_data
        self.mispricing_simulator = mispricing_simulator

        # Get rates DataFrame
        self.rates_df = exchange_rate_data.get_rates().copy()

        # Set transaction costs
        if isinstance(transaction_costs, dict):
            self.transaction_costs = transaction_costs
        else:
            # Use the same cost for all pairs
            self.transaction_costs = {
                'EUR/USD': transaction_costs,
                'USD/JPY': transaction_costs,
                'EUR/JPY': transaction_costs
            }

        # Initialize storage for results
        self.results_df = None

    def detect_opportunities(self):
        """Detect arbitrage opportunities based on rate discrepancies"""
        # Ensure required columns exist
        required_columns = ['EUR/USD', 'USD/JPY', 'EUR/JPY', 'EUR/JPY_implied']
        for col in required_columns:
            if col not in self.rates_df.columns:
                if col == 'EUR/JPY_implied':
                    # Calculate implied EUR/JPY if not available
                    self.rates_df['EUR/JPY_implied'] = self.rates_df['EUR/USD'] * self.rates_df['USD/JPY']
                else:
                    raise ValueError(f"Required column '{col}' not found in exchange rate data.")

        # Create results DataFrame
        self.results_df = pd.DataFrame(index=self.rates_df.index)

        # Copy exchange rates to results
        for col in required_columns:
            self.results_df[col] = self.rates_df[col]

        # Calculate mispricing (epsilon)
        if 'Epsilon' not in self.rates_df.columns:
            self.results_df['Epsilon'] = (self.rates_df['EUR/JPY'] / self.rates_df['EUR/JPY_implied']) - 1
        else:
            self.results_df['Epsilon'] = self.rates_df['Epsilon']

        # Detect arbitrage opportunities
        # Case 1: EUR/JPY is overpriced (actual > implied)
        self.results_df['Case_1_Opportunity'] = self.results_df['EUR/JPY'] > self.results_df['EUR/JPY_implied']

        # Case 2: EUR/JPY is underpriced (actual < implied)
        self.results_df['Case_2_Opportunity'] = self.results_df['EUR/JPY'] < self.results_df['EUR/JPY_implied']

        # Any opportunity
        self.results_df['Any_Opportunity'] = (
            self.results_df['Case_1_Opportunity'] |
            self.results_df['Case_2_Opportunity']
        )

        return self.results_df

    def calculate_profits(self):
        """Calculate theoretical profits from identified opportunities"""
        if self.results_df is None:
            self.detect_opportunities()

        # Calculate profit factors
        # Case 1: EUR/JPY is overpriced
        self.results_df['Profit_Factor_Case_1'] = (
            self.results_df['EUR/JPY'] /
            (self.results_df['EUR/USD'] * self.results_df['USD/JPY'])
        )

        # Case 2: EUR/JPY is underpriced
        self.results_df['Profit_Factor_Case_2'] = (
            (self.results_df['EUR/USD'] * self.results_df['USD/JPY']) /
            self.results_df['EUR/JPY']
        )

        # General profit factor (max of the two cases)
        self.results_df['Profit_Factor'] = np.maximum(
            self.results_df['Profit_Factor_Case_1'],
            self.results_df['Profit_Factor_Case_2']
        )

        # Calculate percentage profit
        self.results_df['Percentage_Profit'] = (self.results_df['Profit_Factor'] - 1) * 100

        # Simplified profit factor based on epsilon
        self.results_df['Profit_Factor_Epsilon'] = np.maximum(
            1 + self.results_df['Epsilon'],
            1 / (1 + self.results_df['Epsilon'])
        )

        # Calculate percentage profit based on epsilon
        self.results_df['Percentage_Profit_Epsilon'] = (self.results_df['Profit_Factor_Epsilon'] - 1) * 100

        return self.results_df

    def analyze_feasibility(self):
        """Analyze feasibility of opportunities considering transaction costs"""
        if self.results_df is None or 'Percentage_Profit' not in self.results_df.columns:
            self.calculate_profits()

        # Calculate total transaction costs for each arbitrage loop
        # Each loop involves 3 trades, one for each currency pair
        total_costs = sum(self.transaction_costs.values()) * 100  # Convert to percentage

        # Determine feasible opportunities (profit > costs)
        self.results_df['Feasible_Opportunity'] = self.results_df['Percentage_Profit'] > total_costs

        # Calculate net profit after costs
        self.results_df['Net_Profit'] = self.results_df['Percentage_Profit'] - total_costs

        # Count feasible opportunities
        feasible_count = self.results_df['Feasible_Opportunity'].sum()
        total_count = len(self.results_df)
        feasible_percentage = (feasible_count / total_count) * 100 if total_count > 0 else 0

        # Store summary statistics
        self.feasibility_summary = {
            'total_opportunities': total_count,
            'feasible_opportunities': feasible_count,
            'feasible_percentage': feasible_percentage,
            'average_profit': self.results_df['Percentage_Profit'].mean(),
            'average_net_profit': self.results_df['Net_Profit'][self.results_df['Feasible_Opportunity']].mean()
                if feasible_count > 0 else 0,
            'max_profit': self.results_df['Percentage_Profit'].max(),
            'max_net_profit': self.results_df['Net_Profit'].max(),
            'transaction_costs': total_costs
        }

        return self.results_df

    def get_results(self):
        """Return detection and analysis results"""
        if self.results_df is None:
            self.analyze_feasibility()

        return self.results_df, self.feasibility_summary

In [ ]:
#############################################
# Part 5: Visualization                    #
#############################################

class ArbitrageVisualizer:
    """
    Comprehensive visualization tools for triangular arbitrage analysis

    This class provides functionality to:
    1. Create integrated dashboards of arbitrage results
    2. Generate publication-quality figures for reports
    3. Visualize relationships between different parameters
    """

    def __init__(self, exchange_data, parameter_estimator=None,
                 mispricing_simulator=None, arbitrage_detector=None):
        """
        Initialize the visualizer with data from all components

        Parameters:
        -----------
        exchange_data : ExchangeRateData
            Exchange rate data object
        parameter_estimator : ParameterEstimator, optional
            Parameter estimation object
        mispricing_simulator : MispricingSimulator, optional
            Mispricing simulation object
        arbitrage_detector : ArbitrageDetector, optional
            Arbitrage detection object
        """
        self.exchange_data = exchange_data
        self.parameter_estimator = parameter_estimator
        self.mispricing_simulator = mispricing_simulator
        self.arbitrage_detector = arbitrage_detector

    def create_exchange_rate_dashboard(self, time_period=None):
        """
        Create a dashboard of exchange rate data

        Parameters:
        -----------
        time_period : tuple, optional
            Time period to display (start_date, end_date)
        """
        rates_df = self.exchange_data.get_rates()

        # Filter by time period if specified
        if time_period is not None:
            start_date, end_date = time_period
            rates_df = rates_df.loc[start_date:end_date]

        # Create figure
        fig = plt.figure(figsize=(15, 12))
        gs = GridSpec(3, 2, figure=fig)

        # Plot EUR/USD
        ax1 = fig.add_subplot(gs[0, 0])
        rates_df['EUR/USD'].plot(ax=ax1, color='blue', linewidth=2)
        ax1.set_title('EUR/USD Exchange Rate', fontsize=14)
        ax1.set_ylabel('Rate', fontsize=12)
        ax1.grid(True)

        # Plot USD/JPY
        ax2 = fig.add_subplot(gs[0, 1])
        rates_df['USD/JPY'].plot(ax=ax2, color='green', linewidth=2)
        ax2.set_title('USD/JPY Exchange Rate', fontsize=14)
        ax2.set_ylabel('Rate', fontsize=12)
        ax2.grid(True)

        # Plot EUR/JPY (actual and implied)
        ax3 = fig.add_subplot(gs[1, :])
        rates_df['EUR/JPY'].plot(ax=ax3, color='red', linewidth=2, label='Actual')
        rates_df['EUR/JPY_implied'].plot(ax=ax3, color='orange', linestyle='--',
                                         linewidth=2, label='Implied')
        ax3.set_title('EUR/JPY Exchange Rate (Actual vs Implied)', fontsize=14)
        ax3.set_ylabel('Rate', fontsize=12)
        ax3.legend(fontsize=12)
        ax3.grid(True)

        # Plot mispricing (epsilon)
        ax4 = fig.add_subplot(gs[2, 0])
        rates_df['Epsilon'].plot(ax=ax4, color='purple', linewidth=2)
        ax4.set_title('Mispricing (Epsilon)', fontsize=14)
        ax4.set_ylabel('Epsilon', fontsize=12)
        ax4.set_xlabel('Date', fontsize=12)
        ax4.axhline(y=0, color='black', linestyle='-', alpha=0.3)
        ax4.grid(True)

        # Plot histogram of mispricing
        ax5 = fig.add_subplot(gs[2, 1])
        rates_df['Epsilon'].hist(ax=ax5, bins=30, color='purple', alpha=0.7)
        ax5.set_title('Distribution of Mispricing', fontsize=14)
        ax5.set_xlabel('Epsilon', fontsize=12)
        ax5.set_ylabel('Frequency', fontsize=12)
        ax5.grid(True)

        # Format x-axis dates
        for ax in [ax1, ax2, ax3, ax4]:
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
            ax.xaxis.set_major_locator(mdates.AutoDateLocator())
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

        plt.tight_layout()
        return fig

    def create_arbitrage_dashboard(self, time_period=None):
        """
        Create a dashboard of arbitrage detection results

        Parameters:
        -----------
        time_period : tuple, optional
            Time period to display (start_date, end_date)
        """
        if self.arbitrage_detector is None:
            raise ValueError("Arbitrage detector not provided. Cannot create dashboard.")

        results_df, feasibility_summary = self.arbitrage_detector.get_results()

        # Filter by time period if specified
        if time_period is not None:
            start_date, end_date = time_period
            results_df = results_df.loc[start_date:end_date]

        # Create figure
        fig = plt.figure(figsize=(15, 15))
        gs = GridSpec(4, 2, figure=fig)

        # Plot profit time series
        ax1 = fig.add_subplot(gs[0, :])
        results_df['Percentage_Profit'].plot(ax=ax1, color='green', linewidth=2)
        if 'transaction_costs' in feasibility_summary:
            ax1.axhline(
                y=feasibility_summary['transaction_costs'],
                color='red', linestyle='--', linewidth=2,
                label=f"Transaction Costs: {feasibility_summary['transaction_costs']:.4f}%"
            )
        ax1.set_title('Arbitrage Profit Over Time', fontsize=14)
        ax1.set_ylabel('Profit (%)', fontsize=12)
        ax1.legend(fontsize=12)
        ax1.grid(True)

        # Plot profit distribution
        ax2 = fig.add_subplot(gs[1, 0])
        results_df['Percentage_Profit'].hist(ax=ax2, bins=30, color='green', alpha=0.7)
        if 'transaction_costs' in feasibility_summary:
            ax2.axvline(
                x=feasibility_summary['transaction_costs'],
                color='red', linestyle='--', linewidth=2,
                label=f"Transaction Costs: {feasibility_summary['transaction_costs']:.4f}%"
            )
        ax2.set_title('Distribution of Percentage Profits', fontsize=14)
        ax2.set_xlabel('Profit (%)', fontsize=12)
        ax2.set_ylabel('Frequency', fontsize=12)
        ax2.legend(fontsize=12)
        ax2.grid(True)

        # Plot net profit distribution (feasible opportunities)
        ax3 = fig.add_subplot(gs[1, 1])
        if 'Net_Profit' in results_df.columns and 'Feasible_Opportunity' in results_df.columns:
            feasible_profits = results_df['Net_Profit'][results_df['Feasible_Opportunity']]
            if len(feasible_profits) > 0:
                feasible_profits.hist(ax=ax3, bins=30, color='blue', alpha=0.7)
                ax3.set_title('Distribution of Net Profits (Feasible Opportunities)', fontsize=14)
            else:
                ax3.set_title('No Feasible Opportunities', fontsize=14)
        else:
            ax3.set_title('Net Profit Data Not Available', fontsize=14)
        ax3.set_xlabel('Net Profit (%)', fontsize=12)
        ax3.set_ylabel('Frequency', fontsize=12)
        ax3.grid(True)

        # Plot opportunity frequency
        ax4 = fig.add_subplot(gs[2, 0])
        opportunity_counts = {
            'Case 1\n(EUR/JPY Overpriced)': results_df['Case_1_Opportunity'].sum(),
            'Case 2\n(EUR/JPY Underpriced)': results_df['Case_2_Opportunity'].sum(),
            'Any\nOpportunity': results_df['Any_Opportunity'].sum()
        }
        if 'Feasible_Opportunity' in results_df.columns:
            opportunity_counts['Feasible\nOpportunities'] = results_df['Feasible_Opportunity'].sum()

        total_periods = len(results_df)
        opportunity_percentages = {
            k: (v / total_periods) * 100 for k, v in opportunity_counts.items()
        }

        bars = ax4.bar(
            opportunity_percentages.keys(),
            opportunity_percentages.values(),
            color=['blue', 'green', 'orange', 'red'][:len(opportunity_percentages)]
        )

        for bar in bars:
            height = bar.get_height()
            ax4.text(
                bar.get_x() + bar.get_width() / 2.,
                height + 0.5,
                f'{height:.2f}%',
                ha='center', va='bottom'
            )

        ax4.set_title('Frequency of Arbitrage Opportunities', fontsize=14)
        ax4.set_ylabel('Percentage of Periods (%)', fontsize=12)
        ax4.set_ylim(0, max(opportunity_percentages.values()) * 1.2)
        ax4.grid(axis='y', linestyle='--', alpha=0.7)

        # Plot epsilon vs profit scatter
        ax5 = fig.add_subplot(gs[2, 1])
        ax5.scatter(
            results_df['Epsilon'].abs(),
            results_df['Percentage_Profit'],
            alpha=0.5, color='purple'
        )

        z = np.polyfit(results_df['Epsilon'].abs(), results_df['Percentage_Profit'], 1)
        p = np.poly1d(z)
        ax5.plot(
            sorted(results_df['Epsilon'].abs()),
            p(sorted(results_df['Epsilon'].abs())),
            "r--", linewidth=2
        )

        ax5.set_title('Relationship Between |Epsilon| and Profit', fontsize=14)
        ax5.set_xlabel('|Epsilon| (Absolute Mispricing)', fontsize=12)
        ax5.set_ylabel('Profit (%)', fontsize=12)
        ax5.grid(True)

        # Plot feasibility summary table
        ax6 = fig.add_subplot(gs[3, :])

        # Create summary table
        summary_data = {
            'Metric': [
                'Total Opportunities',
                'Feasible Opportunities',
                'Feasible Percentage',
                'Average Profit',
                'Average Net Profit (Feasible)',
                'Max Profit',
                'Max Net Profit',
                'Transaction Costs'
            ],
            'Value': [
                feasibility_summary.get('total_opportunities', 'N/A'),
                feasibility_summary.get('feasible_opportunities', 'N/A'),
                f"{feasibility_summary.get('feasible_percentage', 'N/A'):.2f}%",
                f"{feasibility_summary.get('average_profit', 'N/A'):.4f}%",
                f"{feasibility_summary.get('average_net_profit', 'N/A'):.4f}%",
                f"{feasibility_summary.get('max_profit', 'N/A'):.4f}%",
                f"{feasibility_summary.get('max_net_profit', 'N/A'):.4f}%",
                f"{feasibility_summary.get('transaction_costs', 'N/A'):.4f}%"
            ]
        }

        # Hide axes
        ax6.axis('off')

        # Create table
        table = ax6.table(
            cellText=[[m, v] for m, v in zip(summary_data['Metric'], summary_data['Value'])],
            colLabels=['Metric', 'Value'],
            loc='center',
            cellLoc='center'
        )

        # Style table
        table.auto_set_font_size(False)
        table.set_fontsize(12)
        table.scale(1, 1.5)

        # Format x-axis dates for time series plots
        for ax in [ax1]:
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
            ax.xaxis.set_major_locator(mdates.AutoDateLocator())
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

        plt.tight_layout()
        return fig

In [ ]:
#############################################
# Part 6: Main Execution                   #
#############################################

def run_triangular_arbitrage_simulation(num_periods=1000, transaction_cost=0.0005,
                                        volatility_epsilon=0.001, correlation=-0.84,
                                        initial_rates=None, volatilities=None,
                                        output_pdf=None):
    """
    Run a complete triangular arbitrage simulation

    Parameters:
    -----------
    num_periods : int
        Number of periods for simulation
    transaction_cost : float
        Transaction cost as a percentage (e.g., 0.0005 for 0.05%)
    volatility_epsilon : float
        Volatility of the mispricing term
    correlation : float
        Correlation between EUR/USD and USD/JPY returns
    initial_rates : dict, optional
        Initial exchange rates
    volatilities : dict, optional
        Volatility parameters for exchange rates
    output_pdf : str, optional
        Path to save the report (PDF format)
    """
    print("Starting Triangular Arbitrage Simulation...")
    print(f"Parameters: periods={num_periods}, transaction_cost={transaction_cost*100}%, "
          f"volatility_epsilon={volatility_epsilon}, correlation={correlation}")

    # Default initial rates if not provided
    if initial_rates is None:
        initial_rates = {
            'EUR/USD': 1.13,
            'USD/JPY': 143.0,
            'EUR/JPY': 161.5
        }

    # Default volatilities if not provided
    if volatilities is None:
        volatilities = {
            'EUR/USD': 0.004,  # σEU
            'USD/JPY': 0.0043  # σUJ
        }

    # Step 1: Create exchange rate data
    print("\n1. Generating exchange rate data...")
    exchange_data = ExchangeRateData(
        data_source='historical',
        num_periods=num_periods,
        initial_rates=initial_rates,
        volatilities=volatilities,
        correlation=correlation
    )

    # Step 2: Estimate parameters
    print("\n2. Estimating statistical parameters...")
    estimator = ParameterEstimator(exchange_data)
    parameters = estimator.estimate_all_parameters()

    print("Estimated Parameters:")
    for param_name, param_value in parameters.items():
        print(f"  {param_name}: {param_value}")

    # Step 3: Simulate mispricing
    print("\n3. Simulating mispricing...")
    simulator = MispricingSimulator(
        exchange_data,
        parameters={'volatility_epsilon': volatility_epsilon}
    )

    # Calculate implied rates
    simulator.calculate_implied_rates()

    # Generate mispricing
    simulator.generate_mispricing()

    # Calculate actual rates
    simulator.calculate_actual_rates()

    # Analyze mispricing distribution
    mispricing_stats = simulator.analyze_mispricing_distribution()
    print("Mispricing Distribution Statistics:")
    for stat_name, stat_value in mispricing_stats.items():
        print(f"  {stat_name}: {stat_value}")

    # Step 4: Detect arbitrage opportunities
    print("\n4. Detecting arbitrage opportunities...")
    detector = ArbitrageDetector(
        exchange_data,
        mispricing_simulator=simulator,
        transaction_costs=transaction_cost
    )

    # Detect opportunities
    detector.detect_opportunities()

    # Calculate profits
    detector.calculate_profits()

    # Analyze feasibility
    detector.analyze_feasibility()

    # Get results
    results_df, feasibility_summary = detector.get_results()

    print("Arbitrage Feasibility Summary:")
    for key, value in feasibility_summary.items():
        print(f"  {key}: {value}")

    # Step 5: Create visualizations
    print("\n5. Creating visualizations...")
    visualizer = ArbitrageVisualizer(
        exchange_data,
        parameter_estimator=estimator,
        mispricing_simulator=simulator,
        arbitrage_detector=detector
    )

    # Create exchange rate dashboard
    exchange_dashboard = visualizer.create_exchange_rate_dashboard()
    plt.figure(exchange_dashboard.number)
    plt.show()

    # Create arbitrage dashboard
    arbitrage_dashboard = visualizer.create_arbitrage_dashboard()
    plt.figure(arbitrage_dashboard.number)
    plt.show()

    print("\nSimulation completed successfully!")

    return exchange_data, estimator, simulator, detector, visualizer

# Run the simulation
exchange_data, estimator, simulator, detector, visualizer = run_triangular_arbitrage_simulation(
    num_periods=1000,
    transaction_cost=0.0005,  # 0.05% per trade
    volatility_epsilon=0.001,
    correlation=-0.84
)

## High‑Resolution FX Synthetic Generator & Arbitrage Analytics

The following cells were appended on **2025‑04‑27** to extend the original project:1. Implement a hybrid **second‑by‑second FX generator** that anchors on real 1‑minute data (or falls back to fully synthetic).
2. Demonstrate generation of 10 trading days of data.
3. Evaluate **triangular‑arbitrage probability** for the historically interesting window **22 – 27 Apr 2025**.
4. Provide a **rolling scanner** that searches the past year for other high‑probability windows.

In [ ]:
text = 'yfinance';
import importlib, subprocess, sys
if importlib.util.find_spec(text) is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', text, 'pandas_datareader'])

In [ ]:

# --- synthetic_fx_data.py (embedded) ---
from __future__ import annotations
import numpy as np, pandas as pd
from datetime import datetime, timedelta, timezone
from typing import Literal, Tuple
try:
    import yfinance as yf  # type: ignore
except ImportError:
    yf = None  # offline

def _chol(vol: Tuple[float, float], corr: float):
    cov = np.array([[vol[0]**2, corr*vol[0]*vol[1]],
                    [corr*vol[0]*vol[1], vol[1]**2]])
    return np.linalg.cholesky(cov)

def _bridge(series: pd.Series, seconds: int, vol: float) -> pd.Series:
    logp = np.log(series.values)
    per_min = len(logp) - 1
    sec_per_min = seconds // per_min
    out = np.empty(seconds + 1); idx = 0
    rng = np.random.default_rng()
    for i in range(per_min):
        a, b = logp[i], logp[i+1]
        for s in range(sec_per_min):
            t = (s+1)/sec_per_min
            mean = (1-t)*a + t*b
            var = vol**2 * t * (1-t)
            out[idx] = rng.normal(mean, np.sqrt(max(var,0)))
            idx += 1
    out[-1] = logp[-1]
    return pd.Series(np.exp(out), index=range(seconds+1))

def generate_fx_dataset(*, days:int=10, mode:Literal['historical','synthetic']='historical',
                        start_rate_eur_usd=1.07, start_rate_usd_jpy=155.0,
                        vol:Tuple[float,float]=(5e-5,5e-5), corr:float=0.25,
                        kappa=0.05, sigma_m=2e-4, seed:int|None=None):
    """Return DataFrame with second‑level EUR/USD, USD/JPY, implied & actual EUR/JPY."""
    if seed is not None:
        np.random.seed(seed)
    seconds = days*24*3600
    if mode=='historical' and yf is not None:
        try:
            end = datetime.now(timezone.utc)
            start = end - timedelta(days=days)
            raw = yf.download(['EURUSD=X','USDJPY=X'], start=start, end=end,
                              interval='1m', progress=False, timeout=60)
            if raw.empty:
                raise ValueError('no data')
            minute = raw['Close'].dropna().ffill()
            minute.columns = ['EUR/USD','USD/JPY']
            secs_tot = len(minute.index)*60
            eur = _bridge(minute['EUR/USD'], secs_tot, vol[0]).values
            jpy = _bridge(minute['USD/JPY'], secs_tot, vol[1]).values
        except Exception as e:
            print('[generate_fx_dataset] fallback synthetic:', e)
            mode='synthetic'
    if mode=='synthetic':
        L=_chol(vol,corr)
        dW = np.random.normal(size=(seconds,2))@L.T
        eur=np.empty(seconds+1); jpy=np.empty(seconds+1)
        eur[0]=start_rate_eur_usd; jpy[0]=start_rate_usd_jpy
        for t in range(seconds):
            eur[t+1]=eur[t]*np.exp(-0.5*vol[0]**2 + dW[t,0])
            jpy[t+1]=jpy[t]*np.exp(-0.5*vol[1]**2 + dW[t,1])
    implied = eur*jpy
    m=np.zeros_like(implied)
    for t in range(1,len(m)):
        m[t]=m[t-1]+kappa*(-m[t-1])+sigma_m*np.random.normal()
    actual = implied*(1+m)
    idx=pd.date_range(start=datetime.now(timezone.utc), periods=len(eur), freq='S')
    return pd.DataFrame({'EUR/USD':eur,'USD/JPY':jpy,
                         'EUR/JPY_implied':implied,'EUR/JPY_actual':actual},
                        index=idx)

print('Synthetic module loaded ✓')

In [ ]:
%matplotlib inline
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter

# ── 0) FIXED SEED ─────────────────────────────────────────────────────────────
SEED = 12345
rng  = np.random.default_rng(SEED)

# ── 1) Bridge helper: linear‐in‐log plus noise between two anchor points
def _bridge(series: pd.Series, steps: int, vol: float, rng: np.random.Generator) -> np.ndarray:
    logp = np.log(series.values)
    out  = np.empty(steps + 1)
    for i in range(steps):
        t    = (i + 1) / steps
        mean = (1 - t)*logp[0] + t*logp[-1]
        var  = vol**2 * t * (1 - t)
        out[i] = rng.normal(loc=mean, scale=np.sqrt(max(var, 0)))
    out[-1] = logp[-1]
    return np.exp(out)

# ── 2) USER-CHANGEABLE SETTINGS ────────────────────────────────────────────────
DATE     = "2025-04-23"   # date to analyze
START_H  = 14             # 14:00
END_H    = 15             # 15:00
FREQ     = 'min'          # 'min' = 1-minute bars, 's' = 1-second bars
VOL      = 5e-5           # interpolation volatility
KAPPA    = 0.05           # mean-reversion speed for mispricing
SIGMA_M  = 2e-4           # mispricing noise per step

# ── Derived parameters ───────────────────────────────────────────────────────
POINTS_H = (END_H - START_H) * (60 if FREQ=='min' else 3600)
XL_FMT   = "%H:%M" if FREQ=='min' else "%H:%M:%S"
RES_LBL  = "minute" if FREQ=='min' else "second"

# ── 3) Fetch & parse JSON cache ───────────────────────────────────────────────
url  = "https://palashsharma.com/exchange_rate_cache.json"
resp = requests.get(url, timeout=10)
resp.raise_for_status()
raw  = resp.json()

df = (
    pd.DataFrame.from_dict(raw, orient="index")
      .rename_axis("date")
      .reset_index()
      .assign(date=lambda d: pd.to_datetime(d["date"], format="%Y/%m/%d"))
      .set_index("date")
      .sort_index()
)

day0 = pd.to_datetime(DATE)
day1 = day0 + pd.Timedelta(days=1)

# ── 4) Grab any JSON points between day0 and day1 ─────────────────────────────
window     = df.loc[day0:day1, ["EUR/USD","USD/JPY"]]
use_bridge = len(window) >= 2

# ── 5) Build high-res FX + mispricing ────────────────────────────────────────
if use_bridge:
    print("Using bridge interpolation with fixed seed")
    total_steps = 24*60 if FREQ=='min' else 24*3600
    pts         = window.iloc[[0, -1]]
    eur_full    = _bridge(pts["EUR/USD"], total_steps, VOL, rng)
    jpy_full    = _bridge(pts["USD/JPY"], total_steps, VOL, rng)
    implied     = eur_full * jpy_full

    # OU‐style mispricing in fractional units, then convert to bp
    m = np.zeros_like(implied)
    for t in range(1, len(m)):
        # use the SAME rng for each draw
        m[t] = m[t-1] + KAPPA * (-m[t-1]) + SIGMA_M * rng.normal()

    actual   = implied * (1 + m)
    times    = pd.date_range(start=pts.index[0], periods=len(eur_full), freq=FREQ)

    fx_full  = pd.DataFrame({
        "EUR/USD": eur_full,
        "USD/JPY": jpy_full,
        "Implied": implied,
        "Actual": actual,
        "Mispricing": m * 10000  # basis points
    }, index=times)

    # slice 14:00–15:00
    start_ts = day0 + pd.Timedelta(hours=START_H)
    end_ts   = day0 + pd.Timedelta(hours=END_H)
    fx_win   = fx_full.loc[start_ts:end_ts]

else:
    print("Fallback GBM + OU simulation with fixed seed")
    if day0 not in df.index:
        raise RuntimeError(f"No JSON rate for {DATE} midnight to seed simulation.")
    seed_eur = df.at[day0, "EUR/USD"]
    seed_jpy = df.at[day0, "USD/JPY"]

    eur = np.empty(POINTS_H)
    jpy = np.empty(POINTS_H)
    m   = np.zeros(POINTS_H)

    eur[0], jpy[0] = seed_eur, seed_jpy
    for i in range(POINTS_H-1):
        eur[i+1] = eur[i] * np.exp(-0.5*VOL**2 + VOL * rng.normal())
        jpy[i+1] = jpy[i] * np.exp(-0.5*VOL**2 + VOL * rng.normal())
        m[i+1]   = m[i] + KAPPA*(-m[i]) + SIGMA_M * rng.normal()

    implied  = eur * jpy
    actual   = implied * (1 + m)
    start_ts = day0 + pd.Timedelta(hours=START_H)
    times    = pd.date_range(start=start_ts, periods=POINTS_H, freq=FREQ)
    fx_win   = pd.DataFrame({
        "EUR/USD": eur,
        "USD/JPY": jpy,
        "Implied": implied,
        "Actual": actual,
        "Mispricing": m * 10000
    }, index=times)

# ── 6) Plot: top panel prices, bottom panel mispricing ────────────────────────
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(12, 8), sharex=True,
    gridspec_kw={"height_ratios":[2,1]}
)

ax3, = ax1.twinx(),
ln1, = ax1.plot(
    fx_win.index, fx_win["EUR/USD"],
    label="EUR/USD",
    color="tab:blue"          # specify blue
)
ln2, = ax3.plot(
    fx_win.index, fx_win["USD/JPY"],
    label="USD/JPY",
    color="tab:orange"        # specify orange
)

ax1.set_ylabel("EUR/USD")
ax3.set_ylabel("USD/JPY")
ax1.set_title(f"FX & Mispricing on {DATE} {START_H:02d}:00–{END_H:02d}:00 ({RES_LBL}-level)")
ax2.plot(
    fx_win.index, fx_win["Mispricing"],
    label="Mispricing (bp)",
    color="tab:green"         # specify green
)
ax2.axhline(0, ls="--")
ax2.set_ylabel("Mispricing (bp)")
ax2.set_xlabel("Time")
ax2.legend(loc="upper left")

ax1.xaxis.set_major_formatter(DateFormatter(XL_FMT))
ax1.legend([ln1, ln2], [ln1.get_label(), ln2.get_label()], loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

from ipywidgets import DatePicker, FloatSlider, Button, VBox, HBox, HTML as WHTML
from IPython.display import display, clear_output, HTML as DHTML

plt.style.use("default")
plt.rcParams["figure.figsize"] = (9, 4.5)

# ─── define data bounds ─────────────────────────────────────────────────────
data_start = pd.Timestamp("2003-12-01")
data_end   = pd.Timestamp("2025-04-28")

def plot_mispricing(start, end, thr_bp):
    # … your existing fetch + plot logic …
    url = "https://palashsharma.com/exchange_rate_cache.json"
    resp = requests.get(url, timeout=10); resp.raise_for_status()
    df = (
        pd.DataFrame.from_dict(resp.json(), orient="index")
          .rename_axis("Date")
          .reset_index()
          .assign(Date=lambda d: pd.to_datetime(d["Date"], format="%Y/%m/%d"))
          .sort_values("Date")
          .set_index("Date")
    ).loc[start:end]

    if df.empty:
        print("No data for that date range.")
        return

    mid     = df[["EUR/USD","USD/JPY","EUR/JPY"]].ffill()
    implied = mid["EUR/USD"] * mid["USD/JPY"]
    mis     = mid["EUR/JPY"] / implied - 1
    thr_frac = thr_bp / 10_000
    prob    = (mis.abs() >= thr_frac).mean()

    ax = (mis * 10_000).hist(bins=200, edgecolor="k", lw=0.3)
    ax.axhline( thr_bp, ls="--")
    ax.axhline(-thr_bp, ls="--")
    ax.set_title(f"EUR/JPY mis-pricing   {start:%Y-%m-%d} → {end:%Y-%m-%d}")
    ax.set_xlabel("Mis-pricing (bp)")
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()

    stats = (mis * 10_000).describe([0.05,0.25,0.5,0.75,0.95])
    print("\n", stats[["min","5%","25%","50%","75%","95%","max"]]
          .to_string(float_format="%.3f"))
    print(f"\nP(|mis| ≥ {thr_bp:.2f} bp) = {prob*100:.2f}%\n")


today = pd.Timestamp.today().normalize()

# ─── build widgets ─────────────────────────────────────────────────────────
instr = WHTML(
    value=(
        f"<b>Instructions:</b> Select start/end dates (available {data_start:%Y-%m-%d} – "
        f"{data_end:%Y-%m-%d}), pick a threshold (bp), then click <i>Plot</i>."
    )
)
w_start = DatePicker(
    description="Start",
    value=data_start,          # ← default to 2003-12-01
    min=data_start,
    max=data_end
)
w_end = DatePicker(
    description="End",
    value=min(data_end, today),
    min=data_start,
    max=data_end
)
w_thr = FloatSlider(
    description="Threshold (bp)",
    value=5, min=0.1, max=50, step=0.1,
    readout_format=".1f"
)
btn = Button(description="Plot", button_style="success")
msg = WHTML()

def on_click(_):
    msg.value = ""
    # convert to Timestamps right away
    start = pd.Timestamp(w_start.value)
    end   = pd.Timestamp(w_end.value)
    thr   = w_thr.value

    # presence check
    if w_start.value is None or w_end.value is None:
        msg.value = "<span style='color:red'>Please pick both start and end dates.</span>"
        return
    # bounds check
    if start < data_start or end > data_end:
        msg.value = (
            f"<span style='color:red'>Dates must be between "
            f"{data_start:%Y-%m-%d} and {data_end:%Y-%m-%d}.</span>"
        )
        return
    # order check
    if start > end:
        msg.value = "<span style='color:red'>Start date must be on or before End date.</span>"
        return
    # same-day warning
    if start == end:
        msg.value = (
            "<span style='color:orange'>You've chosen the same day—histogram may have very few points.</span>"
        )

    # redraw + breaker + plot
    clear_output(wait=True)
    display(ui)
    display(DHTML("<hr style='border:1px solid #ccc'>"))
    plot_mispricing(start, end, thr)

btn.on_click(on_click)

ui = VBox([instr, HBox([w_start, w_end, w_thr, btn]), msg])
display(ui)